# Kansas City Pipe & Supply — From Agent to Production
### Notebook 1 of the Deploy & Monitor arc: *build it and try it*

**The scenario:** A deep freeze hits Kansas City. Pipes burst. Every contractor in town is asking the counter for prices, stock, and delivery dates at the same time. We'll build an agent to answer them, then (in the next steps) ship it, scale it, and watch it.

| Step | Where | Question it answers |
|---|---|---|
| **1. Build & try** *(this notebook)* | Colab | Does the agent work for one customer? |
| 2. Deploy | Cloud Run | Can anyone call it, safely? |
| 3. Scale | Cloud Run + load test | What happens when 50 contractors call at once? |
| 4. Managed runtime | Vertex AI Agent Engine | What if Google runs the box for us? |
| 5. Observe & evaluate | LangSmith | Is it still giving *correct* answers? |

## Setup

Add these in Colab **Secrets** (🔑 icon in the left sidebar) and toggle notebook access on:
- `ANTHROPIC_API_KEY` — required
- `LANGSMITH_API_KEY` — optional; if present, every run is traced (we'll use this in Block 2)

> ⚠️ **If the install cell prints import errors or version warnings:** *Runtime ▸ Restart session*, then run the cells again from the top (skip the install cell).

In [ ]:
%pip install -q langgraph langchain-anthropic pandas

In [ ]:
import os

def _load(name, required=True):
    """Colab Secrets first, then environment variables."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = os.getenv(name)
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(f"Missing secret: {name}. Add it in Colab Secrets (🔑) and enable notebook access.")
    return bool(value)

_load("ANTHROPIC_API_KEY")

# Optional: point at a different model without touching code (Block 2 compares models this way)
os.environ.setdefault("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")

if _load("LANGSMITH_API_KEY", required=False):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "plumbing-agent-colab"
    print("LangSmith tracing ON  → project: plumbing-agent-colab")
else:
    print("LangSmith tracing OFF (no LANGSMITH_API_KEY) — fine for today")

print("Model:", os.environ["ANTHROPIC_MODEL"])

## The agent is one file

The next cell **writes** `plumbing_agent.py` to disk instead of just running code. That's deliberate: this exact file is what we'll copy into the container for Cloud Run. Notebook and production run the same code, so they can't drift apart.

Skim it for three things:
1. **`CATALOG`** — our "database" is a Python dict. In real life it's an inventory API.
2. **Three nodes:** `extract` (LLM), `lookup` (plain Python), `respond` (LLM).
3. **Prices, stock, and discounts are computed in Python, never by the LLM.**

In [ ]:
%%writefile plumbing_agent.py
"""
plumbing_agent.py — Kansas City Pipe & Supply: counter-desk quoting agent.

ONE file, used three ways:
  1. Imported in the Colab notebook  (build + try it)
  2. Imported by main.py in Cloud Run (deploy + scale)
  3. Traced/evaluated by LangSmith    (monitor + evaluate)

Graph:  START -> extract -> (lookup) -> respond -> END
  extract  : LLM reads the customer message, picks SKUs + quantities (structured output)
  lookup   : plain Python — stock, price, delivery. No LLM. Deterministic = testable.
  respond  : LLM writes the reply using ONLY the lookup facts.

Env vars:
  ANTHROPIC_API_KEY   required
  ANTHROPIC_MODEL     optional, default claude-haiku-4-5-20251001
  ANTHROPIC_MAX_RETRIES optional, default 2 (set 0 to see raw 429s in the load test)
  ANTHROPIC_TEMPERATURE optional, unset by default (some newer models reject it)
  LANGSMITH_TRACING / LANGSMITH_API_KEY / LANGSMITH_PROJECT  optional (Block 2)
"""

from __future__ import annotations

import os
import time
from typing import Literal, Optional, TypedDict

from langgraph.graph import END, START, StateGraph
from langsmith import traceable
from pydantic import BaseModel, Field

DEFAULT_MODEL = "claude-haiku-4-5-20251001"

# ---------------------------------------------------------------------------
# 1. The "database" — a dict. In production this is an inventory API or SQL.
# ---------------------------------------------------------------------------
CATALOG: dict[str, dict] = {
    "PEX-12-ELB":   {"name": '1/2" PEX crimp elbow, 90°, brass',        "price": 1.89,  "stock": 600},
    "PEX-34-ELB":   {"name": '3/4" PEX crimp elbow, 90°, brass',        "price": 2.79,  "stock": 25},
    "PEX-12-TEE":   {"name": '1/2" PEX crimp tee, brass',               "price": 2.29,  "stock": 400},
    "PEX-34-TEE":   {"name": '3/4" PEX crimp tee, brass',               "price": 3.49,  "stock": 150},
    "PEX-12-100R":  {"name": '1/2" PEX-A pipe, 100 ft roll, red (hot)', "price": 54.00, "stock": 40},
    "PEX-12-100B":  {"name": '1/2" PEX-A pipe, 100 ft roll, blue (cold)', "price": 54.00, "stock": 38},
    "PEX-34-100R":  {"name": '3/4" PEX-A pipe, 100 ft roll, red (hot)', "price": 89.00, "stock": 12},
    "RING-12-100":  {"name": '1/2" copper crimp rings, bag of 100',     "price": 18.50, "stock": 80},
    "TOOL-CRIMP":   {"name": 'PEX crimp tool, 1/2" and 3/4" jaws',      "price": 64.99, "stock": 9},
    "HEAT-CABLE-6": {"name": "Self-regulating pipe heat cable, 6 ft",   "price": 39.95, "stock": 0},
    "FOAM-12-6":    {"name": '1/2" foam pipe insulation, 6 ft',         "price": 2.49,  "stock": 1200},
    "PTC-34-CPL":   {"name": '3/4" push-to-connect coupling',           "price": 9.98,  "stock": 60},
    "VALVE-34":     {"name": '3/4" brass ball valve, full port',        "price": 14.75, "stock": 45},
}

BULK_TIERS = [(100, 0.10), (50, 0.05)]  # (min qty, discount) — checked top-down


# ---------------------------------------------------------------------------
# 2. Deterministic tools. @traceable makes each one its own span in LangSmith.
# ---------------------------------------------------------------------------
@traceable(name="check_stock")
def check_stock(sku: str, qty: int) -> dict:
    on_hand = CATALOG[sku]["stock"]
    ship_now = min(on_hand, qty)
    return {"on_hand": on_hand, "ship_now": ship_now, "backorder": qty - ship_now}


@traceable(name="quote_price")
def quote_price(sku: str, qty: int) -> dict:
    unit = CATALOG[sku]["price"]
    discount = next((d for min_q, d in BULK_TIERS if qty >= min_q), 0.0)
    total = round(unit * qty * (1 - discount), 2)
    return {"unit_price": unit, "discount": discount, "line_total": total}


@traceable(name="delivery_estimate")
def delivery_estimate(ship_now: int, backorder: int) -> str:
    if backorder == 0:
        return "next business day"
    if ship_now == 0:
        return "backordered — about 5 business days"
    return f"{ship_now} next business day, remaining {backorder} in about 5 business days"


# ---------------------------------------------------------------------------
# 3. Structured output the extract step must return.
# ---------------------------------------------------------------------------
class LineRequest(BaseModel):
    sku: Optional[str] = Field(description="Exact SKU from the catalog, or null if we don't carry it")
    description: str = Field(description="What the customer asked for, in their words")
    quantity: int = Field(ge=1, description="Quantity requested; 1 if not stated")


class Extraction(BaseModel):
    intent: Literal["order_inquiry", "product_question", "out_of_scope"]
    items: list[LineRequest] = Field(default_factory=list)


class AgentState(TypedDict, total=False):
    message: str
    intent: str
    items: list[dict]
    lines: list[dict]
    order_total: float
    reply: str
    input_tokens: int
    output_tokens: int


# ---------------------------------------------------------------------------
# 4. LLM plumbing (lazy, so importing this file never needs an API key).
# ---------------------------------------------------------------------------
_llm = None


def get_llm():
    global _llm
    if _llm is None:
        from langchain_anthropic import ChatAnthropic

        kwargs = dict(
            model=os.getenv("ANTHROPIC_MODEL", DEFAULT_MODEL),
            max_tokens=800,
            max_retries=int(os.getenv("ANTHROPIC_MAX_RETRIES", "2")),
        )
        # Some newer Claude models reject `temperature`, so it's opt-in only.
        if os.getenv("ANTHROPIC_TEMPERATURE"):
            kwargs["temperature"] = float(os.environ["ANTHROPIC_TEMPERATURE"])
        _llm = ChatAnthropic(**kwargs)
    return _llm


def _catalog_text() -> str:
    return "\n".join(f"{sku}: {v['name']}" for sku, v in CATALOG.items())


EXTRACT_PROMPT = """You are the order desk at Kansas City Pipe & Supply.
Read the customer message and identify each product they want and how many.
Map each product to an exact SKU from this catalog. If we don't carry it, set sku to null.

CATALOG:
{catalog}

Intent rules:
- order_inquiry: they want price, availability, or to buy specific items
- product_question: they ask whether we carry something or which item fits
- out_of_scope: anything unrelated to buying plumbing supplies"""

REPLY_PROMPT = """You are the order desk at Kansas City Pipe & Supply. Write a short, friendly reply
to the customer. Use ONLY the facts in ORDER LINES — never invent prices, stock, or dates.
Mention bulk discounts when applied. For items we don't carry, say so plainly.
Plain text, no markdown tables, under 120 words.

CUSTOMER MESSAGE:
{message}

ORDER LINES:
{lines}

ORDER TOTAL: ${total:.2f}"""

OUT_OF_SCOPE_REPLY = (
    "Thanks for reaching out! I can help with plumbing supplies — pricing, stock, "
    "and delivery. What can I look up for you?"
)


def _tokens(msg) -> tuple[int, int]:
    u = getattr(msg, "usage_metadata", None) or {}
    return u.get("input_tokens", 0), u.get("output_tokens", 0)


# ---------------------------------------------------------------------------
# 5. Graph nodes.
# ---------------------------------------------------------------------------
def extract(state: AgentState) -> AgentState:
    llm = get_llm().with_structured_output(Extraction, include_raw=True)
    result = llm.invoke([
        ("system", EXTRACT_PROMPT.format(catalog=_catalog_text())),
        ("human", state["message"]),
    ])
    parsed: Extraction = result["parsed"]
    tin, tout = _tokens(result["raw"])
    return {
        "intent": parsed.intent,
        "items": [i.model_dump() for i in parsed.items],
        "input_tokens": state.get("input_tokens", 0) + tin,
        "output_tokens": state.get("output_tokens", 0) + tout,
    }


def lookup(state: AgentState) -> AgentState:
    lines = []
    for item in state.get("items", []):
        sku, qty = item.get("sku"), item["quantity"]
        if sku not in CATALOG:
            lines.append({"sku": None, "requested": item["description"], "quantity": qty,
                          "status": "not carried"})
            continue
        stock = check_stock(sku, qty)
        price = quote_price(sku, qty)
        lines.append({
            "sku": sku, "name": CATALOG[sku]["name"], "quantity": qty, **price, **stock,
            "delivery": delivery_estimate(stock["ship_now"], stock["backorder"]),
        })
    total = round(sum(l.get("line_total", 0) for l in lines), 2)
    return {"lines": lines, "order_total": total}


def respond(state: AgentState) -> AgentState:
    if state.get("intent") == "out_of_scope":
        return {"reply": OUT_OF_SCOPE_REPLY}
    msg = get_llm().invoke(REPLY_PROMPT.format(
        message=state["message"],
        lines="\n".join(str(l) for l in state.get("lines", [])) or "(none)",
        total=state.get("order_total", 0.0),
    ))
    tin, tout = _tokens(msg)
    return {
        "reply": msg.content if isinstance(msg.content, str) else str(msg.content),
        "input_tokens": state.get("input_tokens", 0) + tin,
        "output_tokens": state.get("output_tokens", 0) + tout,
    }


def route_after_extract(state: AgentState) -> str:
    return "respond" if state.get("intent") == "out_of_scope" else "lookup"


def build_graph():
    g = StateGraph(AgentState)
    g.add_node("extract", extract)
    g.add_node("lookup", lookup)
    g.add_node("respond", respond)
    g.add_edge(START, "extract")
    g.add_conditional_edges("extract", route_after_extract, {"lookup": "lookup", "respond": "respond"})
    g.add_edge("lookup", "respond")
    g.add_edge("respond", END)
    return g.compile()


graph = build_graph()


def run_agent(message: str) -> dict:
    """Convenience wrapper: one customer message in, one result dict out (with timing)."""
    t0 = time.perf_counter()
    out = graph.invoke({"message": message})
    out["latency_ms"] = round((time.perf_counter() - t0) * 1000)
    out["model"] = os.getenv("ANTHROPIC_MODEL", DEFAULT_MODEL)
    return out


In [ ]:
import importlib, pandas as pd
import plumbing_agent as pa
importlib.reload(pa)   # pick up edits if you re-run %%writefile

pd.DataFrame.from_dict(pa.CATALOG, orient="index").rename_axis("sku")

### What the graph looks like

`extract` decides the route: an out-of-scope question skips the inventory lookup entirely.

In [ ]:
from IPython.display import Image, display
try:
    display(Image(pa.graph.get_graph().draw_mermaid_png()))
except Exception:
    print(pa.graph.get_graph().draw_mermaid())   # offline fallback: paste into mermaid.live

## First customer

A plumber with a burst pipe. Note this order asks for **more 3/4" elbows than we have in stock** — watch how the answer handles it.

In [ ]:
result = pa.run_agent(
    "Pipe burst in my basement. Need 40 of the 3/4 inch PEX elbows and a 100 ft roll "
    "of 3/4 red PEX. When can I get them?"
)

print(result["reply"], "\n")
display(pd.DataFrame(result["lines"])[["sku", "quantity", "unit_price", "discount", "line_total", "ship_now", "backorder", "delivery"]])
print(f"Order total: ${result['order_total']:.2f}")
print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out   Latency: {result['latency_ms']} ms   Model: {result['model']}")

## A morning at the counter

Five customers, five different paths through the same agent:

| Message | What should happen |
|---|---|
| 120 half-inch tees | Bulk tier → 10% off |
| Heat tape × 3 | Sold out → backorder |
| 1" copper sweat elbows | We don't carry it → say so, don't invent a SKU |
| 60 push-to-connect couplings | Bulk tier → 5% off |
| "What's the weather tomorrow?" | Out of scope → polite redirect, no inventory lookup |

In [ ]:
customers = [
    "Price on 120 half-inch crimp tees?",
    "Do you have heat tape? Need 3 for a crawl space.",
    "Got any 1 inch copper sweat elbows?",
    "How much for 60 of the 3/4 push-to-connect couplings?",
    "What's the weather supposed to be tomorrow?",
]

rows = []
for msg in customers:
    r = pa.run_agent(msg)
    rows.append({"message": msg, "intent": r["intent"],
                 "skus": ", ".join(str(l["sku"]) for l in r.get("lines", [])) or "—",
                 "total": r.get("order_total", 0.0),
                 "tokens": r["input_tokens"] + r["output_tokens"],
                 "latency_ms": r["latency_ms"]})
    print(f"▶ {msg}\n  {r['reply']}\n")

pd.DataFrame(rows)

### Design point for PMs: where the LLM is, and where it isn't

| Step | Who does it | Why |
|---|---|---|
| Understand "3/4 red PEX, 100 ft" → `PEX-34-100R` | **LLM** | Messy human language needs judgment |
| Stock, price, bulk discount, delivery date | **Python** | Must be exact, auditable, and identical every time |
| Write a friendly reply from those facts | **LLM** | Tone and phrasing |

This split pays off twice later:
- **Scaling:** only two of the three steps cost tokens or wait on the model provider.
- **Evaluation:** SKU, quantity, and total have *right answers* we can check automatically. Only the reply's tone needs a judge.

## Why this notebook isn't production (yet)

| Concern | In this notebook | What production needs | Where we fix it |
|---|---|---|---|
| **Secrets** | Colab `userdata` | A secrets manager, injected at runtime | Cloud Run + Secret Manager |
| **Access** | Only you, in your browser | An HTTP endpoint with authentication | FastAPI + Cloud Run IAM |
| **Concurrency** | One customer at a time | Many at once | Cloud Run `--concurrency` / `--max-instances` |
| **Limits** | You'll never hit them | The model provider's rate limit is the real ceiling | Quota, backoff, queues, gateway |
| **Visibility** | `print()` | Traces, metrics, quality checks | LangSmith (Block 2) |

The cell below is a 20-second preview of the concurrency row.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

batch = customers[:4]

t0 = time.perf_counter()
for m in batch:
    pa.run_agent(m)
sequential = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(pa.run_agent, batch))
parallel = time.perf_counter() - t0

print(f"4 customers one at a time:  {sequential:.1f}s")
print(f"4 customers at once:        {parallel:.1f}s")
print("\nThe agent spends most of its time WAITING on the model provider, not computing.")
print("That's why one container can serve many requests at once — and why the provider's")
print("rate limit, not our CPU, is usually the first wall we hit.")

## Back-of-envelope cost

Tokens are the unit of cost. Fill in current per-million-token prices for your model from Anthropic's pricing page to turn token counts into dollars.

In [ ]:
avg_tokens = sum(r["tokens"] for r in rows) / len(rows)  # combined in+out per request, from the run above
PRICE_IN_PER_MTOK  = None   # e.g. look up current input price for your ANTHROPIC_MODEL
PRICE_OUT_PER_MTOK = None   # e.g. look up current output price

requests_per_day = 2_000    # a freeze-week surge for one regional supplier
print(f"Average tokens per request: {avg_tokens:,.0f}")
print(f"Tokens per day at {requests_per_day:,} requests: {avg_tokens * requests_per_day:,.0f}")

if PRICE_IN_PER_MTOK and PRICE_OUT_PER_MTOK:
    r = pa.run_agent(customers[0])   # one fresh run to split in vs out
    per_req = (r["input_tokens"] * PRICE_IN_PER_MTOK + r["output_tokens"] * PRICE_OUT_PER_MTOK) / 1e6
    print(f"≈ ${per_req:.4f} per request  →  ≈ ${per_req * requests_per_day:,.2f} per day")
else:
    print("Set the two PRICE_* variables to see dollars.")

## Next: ship it

Everything needed to run this agent as a web service is in the **`deploy/`** folder:

- `plumbing_agent.py` — the file this notebook just wrote, byte for byte
- `main.py` — FastAPI wrapper: `POST /quote`
- `Dockerfile` — the container recipe
- `loadtest.py` — the deep-freeze surge
- `DEPLOY.md` — step-by-step Cloud Shell commands

**Question to carry into the next step:** the agent works for one person. What has to be true for it to work for 200 people at 6 a.m. during a freeze, without anyone's API key leaking?